# 01 - Data preprocessing

Cleans one dataset and writes its stratified train / hold-out split.
CPU only. Run once per dataset. See `PIPELINE.md`, step 1.

In [ ]:
# Colab: upload the raw dataset file. Locally: place it in DATA_DIR.
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    files.upload()

In [ ]:
import io
import os
import subprocess

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

try:
    display
except NameError:
    display = print

## Config

In [ ]:
DATASET  = "diabetes"     # diabetes | yeast | ecoli | pageblocks
DATA_DIR = "."            # folder holding the inputs; outputs are written here too

SEED           = 42
TEST_SIZE      = 0.20     # stratified hold-out, never augmented
MIN_CLASS_SIZE = 10       # classes smaller than this are removed

PRESETS = {
    "diabetes":   dict(file="diabetes_012_health_indicators_BRFSS2015.csv",
                       target="Diabetes_012", drop=[]),
    "yeast":      dict(file="yeast.csv",       target="Target", drop=["Sequence_Name"]),
    "ecoli":      dict(file="ecoli.csv",       target="Target", drop=["Sequence_Name"]),
    "pageblocks": dict(file="page-blocks.csv", target="Target", drop=[]),
}

# UCI ships yeast, ecoli and page blocks as headerless whitespace-separated files.
RAW_SPECS = {
    "yeast": dict(
        raw="yeast.data",
        cols=["Sequence_Name", "mcg", "gvh", "alm", "mit", "erl", "pox", "vac", "nuc", "Target"],
        drop=["Sequence_Name"], labels=None),
    "ecoli": dict(
        raw="ecoli.data",
        cols=["Sequence_Name", "mcg", "gvh", "lip", "chg", "aac", "alm1", "alm2", "Target"],
        drop=["Sequence_Name"], labels=None),
    "pageblocks": dict(
        raw="page-blocks.data.Z",
        cols=["height", "length", "area", "eccen", "p_black", "p_and", "mean_tr",
              "blackpix", "blackand", "wb_trans", "Target"],
        drop=[], labels={1: "text", 2: "horiz_line", 3: "graphic",
                         4: "vert_line", 5: "picture"}),
}

cfg = PRESETS[DATASET]
FILE_PATH  = os.path.join(DATA_DIR, cfg["file"])
TARGET_COL = cfg["target"]
DROP_COLS  = cfg["drop"]

## Step 0 - Convert the raw UCI file (yeast, ecoli, page blocks)

In [ ]:
def convert_raw(dataset):
    """Raw UCI file -> CSV with named columns and Target last. No value is altered."""
    spec = RAW_SPECS[dataset]
    path = os.path.join(DATA_DIR, spec["raw"])
    if path.endswith(".Z"):                      # Unix compress: read through gzip
        text = subprocess.run(["gzip", "-dc", path], capture_output=True,
                              check=True).stdout.decode("utf-8", errors="replace")
    else:
        text = io.open(path, encoding="utf-8", errors="replace").read()

    raw = pd.read_csv(io.StringIO(text), sep=r"\s+", header=None, engine="python")
    assert raw.shape[1] == len(spec["cols"]), "unexpected number of columns"
    raw.columns = spec["cols"]

    out = raw.drop(columns=spec["drop"]) if spec["drop"] else raw.copy()
    if spec["labels"]:
        out["Target"] = out["Target"].map(spec["labels"])
    out = out[[c for c in out.columns if c != "Target"] + ["Target"]]
    out.to_csv(FILE_PATH, index=False)
    print(f"converted {spec['raw']} -> {cfg['file']} ({len(out):,} rows)")


if not os.path.isfile(FILE_PATH):
    if DATASET not in RAW_SPECS:
        raise FileNotFoundError(f"{FILE_PATH} not found - see data/README.md")
    convert_raw(DATASET)

## Step 1 - Load

In [ ]:
df = pd.read_csv(FILE_PATH)
print("loaded:", df.shape)

## Step 2 - Drop identifiers and exact duplicates

In [ ]:
present = [c for c in DROP_COLS if c in df.columns]
if present:
    df = df.drop(columns=present)

before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"duplicates removed: {before - len(df):,} -> {len(df):,} rows")
print("missing values    :", int(df.isnull().sum().sum()))
df = df.dropna().reset_index(drop=True)

## Step 3 - Remove classes below MIN_CLASS_SIZE

In [ ]:
counts = df[TARGET_COL].value_counts()
tiny = counts[counts < MIN_CLASS_SIZE]
if len(tiny):
    print("removing:", ", ".join(f"{k} (n={v})" for k, v in tiny.items()))
    df = df[~df[TARGET_COL].isin(tiny.index)].reset_index(drop=True)
print(f"{df[TARGET_COL].nunique()} classes, {len(df):,} rows")

## Step 4 - Encode the target as 0..K-1

In [ ]:
labels = sorted(df[TARGET_COL].unique())
MAPPING = {lab: i for i, lab in enumerate(labels)}
df[TARGET_COL] = df[TARGET_COL].map(MAPPING).astype(int)
for lab, i in MAPPING.items():
    print(f"  {i} <- {lab}")

## Step 5 - Stratified 80/20 split

In [ ]:
df_train_raw, df_test_raw = train_test_split(
    df, test_size=TEST_SIZE, stratify=df[TARGET_COL], random_state=SEED, shuffle=True)
df_train_raw = df_train_raw.reset_index(drop=True)
df_test_raw = df_test_raw.reset_index(drop=True)

train_counts = df_train_raw[TARGET_COL].value_counts().sort_index()
test_counts = df_test_raw[TARGET_COL].value_counts().sort_index()
summary = pd.DataFrame({"train": train_counts, "test": test_counts})
summary["total"] = summary.sum(axis=1)
display(summary)
print(f"training imbalance ratio: {train_counts.max() / train_counts.min():.2f}")

## Step 6 - Min-Max scaling, fitted on the training rows only

In [ ]:
FEATURES = [c for c in df.columns if c != TARGET_COL]
scaler = MinMaxScaler().fit(df_train_raw[FEATURES])


def scale(part):
    out = pd.DataFrame(scaler.transform(part[FEATURES]), columns=FEATURES)
    out[TARGET_COL] = part[TARGET_COL].values
    return out


df_train = scale(df_train_raw)
df_test = scale(df_test_raw)       # hold-out values outside [0, 1] are left unclipped

_tr = df_train[FEATURES].to_numpy(dtype=float)
assert _tr.min() >= -1e-9 and _tr.max() <= 1 + 1e-9, "training features are not in [0, 1]"
print("train:", df_train.shape, "| test:", df_test.shape)

## Step 7 - Export

In [ ]:
assert df_train.isna().sum().sum() == 0 and df_test.isna().sum().sum() == 0
assert list(df_train.columns) == list(df_test.columns)

TRAIN_FILE = os.path.join(DATA_DIR, f"df_train_{DATASET}.csv")
TEST_FILE  = os.path.join(DATA_DIR, f"df_test_{DATASET}.csv")
df_train.to_csv(TRAIN_FILE, index=False)
df_test.to_csv(TEST_FILE, index=False)
print("written:", TRAIN_FILE, TEST_FILE)

if IN_COLAB:
    files.download(TRAIN_FILE)
    files.download(TEST_FILE)
    files.download(FILE_PATH)     # the converted CSV is the input of notebook 08